In [5]:
!pip install pandas scikit-learn nltk transformers datasets torch joblib

In [6]:
!pip install pandas scikit-learn nltk transformers datasets torch joblib

In [7]:
import pandas as pd

fake1 = pd.read_csv("gossipcop_fake.csv")
real1 = pd.read_csv("gossipcop_real.csv")

fake2 = pd.read_csv("politifact_fake.csv")
real2 = pd.read_csv("politifact_real.csv")

In [8]:
fake1["label"] = 1
fake2["label"] = 1

real1["label"] = 0
real2["label"] = 0

In [9]:
df = pd.concat([fake1, real1, fake2, real2], ignore_index=True)

print(df.shape)
df.head()

(23196, 5)


,id,news_url,title,tweet_ids,label
0,gossipcop-2493749932,www.dailymail.co.uk/tvshowbiz/article-5874213/...,Did Miley Cyrus and Liam Hemsworth secretly ge...,284329075902926848\t284332744559968256\t284335...,1
1,gossipcop-4580247171,hollywoodlife.com/2018/05/05/paris-jackson-car...,Paris Jackson & Cara Delevingne Enjoy Night Ou...,992895508267130880\t992897935418503169\t992899...,1
2,gossipcop-941805037,variety.com/2017/biz/news/tax-march-donald-tru...,Celebrities Join Tax March in Protest of Donal...,853359353532829696\t853359576543920128\t853359...,1
3,gossipcop-2547891536,www.dailymail.co.uk/femail/article-3499192/Do-...,Cindy Crawford's daughter Kaia Gerber wears a ...,988821905196158981\t988824206556172288\t988825...,1
4,gossipcop-5476631226,variety.com/2018/film/news/list-2018-oscar-nom...,Full List of 2018 Oscar Nominations – Variety,955792793632432131\t955795063925301249\t955798...,1


In [10]:
df["content"] = df["title"]

In [11]:
import re
import nltk
from nltk.corpus import stopwords

# Ensure NLTK stopwords are downloaded
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()
    # Filter out stopwords
    words = [w for w in words if w not in stop_words]

    return ' '.join(words)

df['clean_text'] = df['content'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [14]:
from sklearn.svm import SVC

svm_model = SVC(probability=True)

svm_model.fit(X_train_vec, y_train)

SVC(probability=True)

In [15]:
from sklearn.metrics import accuracy_score, classification_report

svm_pred = svm_model.predict(X_test_vec)

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

print(classification_report(y_test, svm_pred))

SVM Accuracy: 0.8467672413793104
              precision    recall  f1-score   support

           0       0.85      0.96      0.90      3492
           1       0.82      0.49      0.61      1148

    accuracy                           0.85      4640
   macro avg       0.83      0.73      0.76      4640
weighted avg       0.84      0.85      0.83      4640



In [16]:
!pip install datasets

In [17]:
from datasets import Dataset

In [18]:
from datasets import Dataset

data = pd.DataFrame({
    "text": df["clean_text"],
    "label": df["label"]
})

train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [19]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/18556 [00:00<?, ? examples/s]

Map:   0%|          | 0/4640 [00:00<?, ? examples/s]

In [21]:
train_dataset = train_dataset.remove_columns(["text", "__index_level_0__"])
test_dataset = test_dataset.remove_columns(["text", "__index_level_0__"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [22]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label"]),
    y=df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print(class_weights)

tensor([0.6650, 2.0153])


In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./results",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    eval_strategy="epoch",
    save_strategy="epoch"
)

In [25]:
from transformers import Trainer

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.401409,0.360907
2,0.302049,0.476908
3,0.221601,0.416522
4,0.155994,0.580718
5,0.119137,0.671061


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5800, training_loss=0.24615588615680564, metrics={'train_runtime': 4388.2349, 'train_samples_per_second': 21.143, 'train_steps_per_second': 1.322, 'total_flos': 1.22057218581504e+16, 'train_loss': 0.24615588615680564, 'epoch': 5.0})

In [27]:
trainer.evaluate()

{'eval_loss': 0.6710605025291443,
 'eval_runtime': 69.0055,
 'eval_samples_per_second': 67.241,
 'eval_steps_per_second': 4.203,
 'epoch': 5.0}

In [28]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(axis=1)

labels = predictions.label_ids

accuracy = accuracy_score(labels, preds)

print("BERT Accuracy:", accuracy)

BERT Accuracy: 0.847198275862069


In [29]:
model.save_pretrained("bert_scam_model")
tokenizer.save_pretrained("bert_scam_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_scam_model/tokenizer_config.json', 'bert_scam_model/tokenizer.json')

In [ ]:
!zip -r bert_scam_model.zip bert_scam_model

from google.colab import files
files.download("bert_scam_model.zip")

updating: bert_scam_model/ (stored 0%)
updating: bert_scam_model/model.safetensors